[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/calculus_optimization/02_taylor_approximation_and_local_models/first_principles.ipynb)

# Topic 02: Taylor Approximation and Local Models

## 1. First-Principles Intuition & Motivation

An optimizer standing at a point $x$ on a loss surface has severely limited vision. It can afford to know $f(x)$, the gradient $\nabla f(x)$, and occasionally some curvature information. From these few numbers it must decide where to step. The only honest way to do that is to build a *model* of the function near $x$ and reason about the model instead.

Taylor's theorem is the contract between model and reality. It says: the polynomial assembled from your local derivative information matches the true function up to an error term, and here is the *exact form* of that error. Optimization theory is largely the art of choosing steps small enough that the error term cannot hurt you — and Taylor's remainder is what "small enough" means quantitatively.

### The model hierarchy

| Order | Local model $m_x(h)$ | Information needed | Algorithm built on it |
|---|---|---|---|
| 0 | $f(x)$ | value | random / grid search |
| 1 | $f(x) + \nabla f(x)^\top h$ | value + gradient | gradient descent |
| 2 | $f(x) + \nabla f(x)^\top h + \frac{1}{2}h^\top \nabla^2 f(x) h$ | value + gradient + curvature | Newton, trust region |

Each extra order buys a better local fit at a higher per-step cost. The engineering trade-off between rows of this table — cheap steps from a crude model versus expensive steps from a faithful one — is the organizing question of practical optimization, resolved differently by SGD, Adam, L-BFGS, and Newton-type methods.

### Why the remainder term is the whole game

Two functions can share value and gradient at $x$ yet diverge immediately afterwards. What separates a trustworthy model from a useless one is a *bound on how fast reality peels away from the model*. That bound comes from controlling one higher derivative:

- Bounded $f''$ (or Hessian) makes the linear model reliable within a computable radius — this becomes the $L$-smoothness assumption and the descent lemma.
- Bounded third derivative makes the quadratic model reliable — this powers Newton's quadratic convergence.

In ML we rarely know these bounds exactly, but the *structure* of the theory still dictates practice: learning-rate warmup, gradient clipping, and trust regions are all remainder-management devices.

### Notation used throughout

- $f: \mathbb{R}^d \to \mathbb{R}$ — the objective (loss); in 1D sections we write $f: \mathbb{R} \to \mathbb{R}$.
- $h \in \mathbb{R}^d$ — the displacement from the expansion point $x$; steps are $y = x + h$.
- $T_k(h)$ — the degree-$k$ Taylor polynomial of $f$ at $x$; $R_k(h) = f(x+h) - T_k(h)$ — its remainder.
- $\nabla f$, $\nabla^2 f$ — gradient and Hessian; $\lVert \cdot \rVert_{\mathrm{op}}$ — the operator (spectral) norm.
- $L$ — the gradient's Lipschitz constant ($L$-smoothness); $M$ — the Hessian's Lipschitz constant.
- $\xi$, $\theta$ — intermediate points produced by mean-value arguments; they exist but are never computed.

## 2. Rigorous Mathematical Definitions & Theorem Statements

**Definition 2.1 (Taylor polynomial).** For $f$ that is $k$ times differentiable at $x$, the degree-$k$ Taylor polynomial of $f$ at $x$ is

$$
T_k(h) = \sum_{j=0}^{k} \frac{f^{(j)}(x)}{j!} h^j = f(x) + f'(x)h + \frac{f''(x)}{2}h^2 + \cdots + \frac{f^{(k)}(x)}{k!}h^k.
$$

**Theorem 2.2 (Taylor's theorem, Lagrange remainder).** If $f$ is $k+1$ times differentiable on an open interval containing $x$ and $x+h$, then there exists $\xi$ strictly between $x$ and $x+h$ such that

$$
f(x+h) = T_k(h) + \frac{f^{(k+1)}(\xi)}{(k+1)!} h^{k+1}.
$$

The case $k = 0$ is the Mean Value Theorem.

**Theorem 2.3 (Multivariate Taylor, second order).** If $f: \mathbb{R}^d \to \mathbb{R}$ is twice continuously differentiable, then for any $x, h$ there exists $\theta \in (0,1)$ such that

$$
f(x+h) = f(x) + \nabla f(x)^\top h + \frac{1}{2} h^\top \nabla^2 f(x + \theta h)\, h.
$$

Evaluating the Hessian at $x$ instead gives the asymptotic form $f(x+h) = f(x) + \nabla f(x)^\top h + \frac{1}{2}h^\top \nabla^2 f(x) h + o(\lVert h \rVert^2)$.

**Definition 2.4 ($L$-smoothness).** A differentiable $f: \mathbb{R}^d \to \mathbb{R}$ is $L$-smooth if its gradient is $L$-Lipschitz:

$$
\lVert \nabla f(x) - \nabla f(y) \rVert_2 \le L \lVert x - y \rVert_2 \quad \text{for all } x, y.
$$

For twice-differentiable $f$, this is equivalent to $\lVert \nabla^2 f(x) \rVert_{\mathrm{op}} \le L$ everywhere, i.e. all Hessian eigenvalues lie in $[-L, L]$.

**Theorem 2.5 (Descent lemma).** If $f$ is $L$-smooth, then for all $x, y$:

$$
f(y) \le f(x) + \nabla f(x)^\top (y - x) + \frac{L}{2} \lVert y - x \rVert_2^2.
$$

The true function is trapped beneath a quadratic built from purely local information — the single most-used inequality in first-order optimization theory.

**Definition 2.6 (Newton step).** At a point $x$ where $\nabla^2 f(x)$ is invertible, the Newton step is the minimizer of the second-order model (when the Hessian is positive definite):

$$
h_{\mathrm{N}} = -\left[ \nabla^2 f(x) \right]^{-1} \nabla f(x), \qquad x_{+} = x + h_{\mathrm{N}}.
$$

**Theorem 2.7 (Newton exactness on quadratics).** If $f(x) = \frac{1}{2}x^\top A x - b^\top x + c$ with $A \succ 0$, then from any starting point a single Newton step lands exactly on the global minimizer $x^\star = A^{-1} b$.

## 3. Step-by-Step Mathematical Proofs & Derivations

### Proof 3.1: Taylor's theorem with Lagrange remainder

*Claim.* (Theorem 2.2) $f(x+h) = T_k(h) + \frac{f^{(k+1)}(\xi)}{(k+1)!}h^{k+1}$ for some $\xi$ between $x$ and $x+h$.

*Proof.* Assume $h \gt 0$ (the case $h \lt 0$ is symmetric); write $b = x + h$. Define for $t \in [x, b]$ the "remaining error at $t$" function

$$
F(t) = f(b) - \sum_{j=0}^{k} \frac{f^{(j)}(t)}{j!}(b - t)^j,
$$

and the comparison function $G(t) = (b - t)^{k+1}$. Note $F(b) = 0 = G(b)$, while $F(x)$ is the remainder $R_k$ we want and $G(x) = h^{k+1}$.

Differentiate $F$ with respect to $t$: the sum telescopes. The $j$-th term contributes $\frac{f^{(j+1)}(t)}{j!}(b-t)^j - \frac{f^{(j)}(t)}{(j-1)!}(b-t)^{j-1}$ (for $j \ge 1$), and consecutive terms cancel pairwise, leaving only

$$
F'(t) = -\frac{f^{(k+1)}(t)}{k!}(b - t)^k, \qquad G'(t) = -(k+1)(b-t)^k.
$$

By the Cauchy Mean Value Theorem applied to $F$ and $G$ on $[x, b]$, there exists $\xi \in (x, b)$ with

$$
\frac{F(x) - F(b)}{G(x) - G(b)} = \frac{F'(\xi)}{G'(\xi)} = \frac{-\frac{f^{(k+1)}(\xi)}{k!}(b-\xi)^k}{-(k+1)(b-\xi)^k} = \frac{f^{(k+1)}(\xi)}{(k+1)!}.
$$

Since $F(b) = G(b) = 0$, the left side is $R_k / h^{k+1}$, giving $R_k = \frac{f^{(k+1)}(\xi)}{(k+1)!} h^{k+1}$. $\blacksquare$

*Consequences.* With $k = 1$: the linear model errs by $\frac{f''(\xi)}{2}h^2 = O(h^2)$. With $k = 2$: the quadratic model errs by $\frac{f'''(\xi)}{6}h^3 = O(h^3)$. These are the slopes 2 and 3 measured on log-log plots in the legacy notebook [`../taylor_approximation.ipynb`](../taylor_approximation.ipynb).

### Proof 3.2: Multivariate second-order Taylor via 1D restriction

*Claim.* (Theorem 2.3) $f(x+h) = f(x) + \nabla f(x)^\top h + \frac{1}{2}h^\top \nabla^2 f(x + \theta h) h$ for some $\theta \in (0,1)$.

*Proof.* Restrict $f$ to the segment: define $g(t) = f(x + t h)$ for $t \in [0, 1]$. By the chain rule,

$$
g'(t) = \nabla f(x + th)^\top h, \qquad g''(t) = h^\top \nabla^2 f(x + th)\, h.
$$

(The second line differentiates each component of $\nabla f(x+th)$ along the direction $h$ again.) Now apply the one-dimensional Taylor theorem (Proof 3.1) with $k = 1$ to $g$ on $[0,1]$: there exists $\theta \in (0,1)$ with

$$
g(1) = g(0) + g'(0) \cdot 1 + \frac{g''(\theta)}{2} \cdot 1^2.
$$

Substituting back $g(1) = f(x+h)$, $g(0) = f(x)$, $g'(0) = \nabla f(x)^\top h$, and $g''(\theta) = h^\top \nabla^2 f(x + \theta h) h$ gives the claim. $\blacksquare$

*Moral.* Every multivariate Taylor statement is a one-variable statement about the restriction of $f$ to a line — a technique worth internalizing, since it converts high-dimensional claims into freshman calculus.

### Proof 3.3: The descent lemma from $L$-smoothness

*Claim.* (Theorem 2.5) If $\lVert \nabla f(u) - \nabla f(v) \rVert_2 \le L\lVert u - v \rVert_2$ for all $u, v$, then $f(y) \le f(x) + \nabla f(x)^\top (y-x) + \frac{L}{2}\lVert y-x \rVert_2^2$.

*Proof.* Write $h = y - x$ and integrate the derivative of $g(t) = f(x + th)$ along the segment (fundamental theorem of calculus):

$$
f(y) - f(x) = \int_0^1 \nabla f(x + th)^\top h \, dt = \nabla f(x)^\top h + \int_0^1 \left( \nabla f(x + th) - \nabla f(x) \right)^\top h \, dt.
$$

Bound the integrand with Cauchy–Schwarz and then the Lipschitz property:

$$
\left( \nabla f(x+th) - \nabla f(x) \right)^\top h \le \lVert \nabla f(x+th) - \nabla f(x) \rVert_2 \lVert h \rVert_2 \le L t \lVert h \rVert_2^2.
$$

Integrate the bound: $\int_0^1 L t \lVert h \rVert_2^2 \, dt = \frac{L}{2}\lVert h \rVert_2^2$. Therefore

$$
f(y) \le f(x) + \nabla f(x)^\top h + \frac{L}{2}\lVert h \rVert_2^2. \qquad \blacksquare
$$

*Why this matters.* Setting $y = x - \eta \nabla f(x)$ turns the inequality into a guaranteed decrease per gradient step (Topic 03, sufficient-decrease lemma). The descent lemma is Taylor's theorem weaponized: it replaces the unknowable $\xi$ of the Lagrange remainder with a *uniform* worst-case constant $L$.

### Proof 3.4: Quadratics are $L$-smooth with $L = \lambda_{\max}$, and Taylor is exact on them

*Claim.* For $f(x) = \frac{1}{2}x^\top A x - b^\top x + c$ with symmetric $A$: (a) the second-order Taylor expansion at any point is exact; (b) $f$ is $L$-smooth with the best constant $L = \max_i \lvert \lambda_i(A) \rvert$.

*Proof.* (a) Compute $\nabla f(x) = Ax - b$ and $\nabla^2 f(x) = A$ (constant). Expand directly:

$$
f(x+h) = \frac{1}{2}(x+h)^\top A (x+h) - b^\top(x+h) + c = f(x) + (Ax - b)^\top h + \frac{1}{2}h^\top A h,
$$

using symmetry of $A$ to merge the cross terms. This is exactly $f(x) + \nabla f(x)^\top h + \frac{1}{2}h^\top \nabla^2 f\, h$ with zero remainder: all third derivatives vanish.

(b) The gradient map is affine: $\nabla f(x) - \nabla f(y) = A(x - y)$. Hence

$$
\lVert \nabla f(x) - \nabla f(y) \rVert_2 = \lVert A(x-y) \rVert_2 \le \lVert A \rVert_{\mathrm{op}} \lVert x - y \rVert_2,
$$

and for symmetric $A$ the operator norm equals the largest absolute eigenvalue $\max_i \lvert \lambda_i \rvert$. Taking $x - y$ along the corresponding eigenvector shows the constant is attained, so no smaller $L$ works. $\blacksquare$

*Bridge to Topic 03.* For the least-squares loss, $A = \frac{2}{n}X^\top X$, so $L = \frac{2}{n}\lambda_{\max}(X^\top X)$ — the quantity that caps the stable learning rate $\eta \lt 2/L$.

### Proof 3.5: Newton's method is exact on quadratics

*Claim.* (Theorem 2.7) For $f(x) = \frac{1}{2}x^\top A x - b^\top x + c$ with $A \succ 0$, one Newton step from any $x_0$ reaches the global minimizer $x^\star = A^{-1}b$.

*Proof.* Since $A \succ 0$, $f$ is strictly convex and the stationarity condition $\nabla f(x) = Ax - b = 0$ has the unique solution $x^\star = A^{-1}b$, which is the global minimizer (Topic 04). The Newton step at $x_0$ uses $\nabla^2 f(x_0) = A$:

$$
x_1 = x_0 - A^{-1}\nabla f(x_0) = x_0 - A^{-1}(A x_0 - b) = x_0 - x_0 + A^{-1}b = x^\star. \qquad \blacksquare
$$

*Why, conceptually.* Newton's method minimizes the second-order model; by Proof 3.4(a) the model *is* the function when $f$ is quadratic, so minimizing the model minimizes the function. Near a nondegenerate minimum of a general smooth $f$, the function is approximately quadratic (Theorem 2.3), which is why Newton converges *quadratically* there: the number of correct digits roughly doubles per iteration, $\lVert x_{t+1} - x^\star \rVert \le C\lVert x_t - x^\star \rVert^2$.

### Proof 3.6: Local quadratic convergence of Newton's method (1D)

*Claim.* Let $f \in C^3$ with $f'(x^\star) = 0$ and $f''(x^\star) \neq 0$. Then Newton's iteration $x_{t+1} = x_t - f'(x_t)/f''(x_t)$ satisfies, for $x_t$ near $x^\star$, $\lvert x_{t+1} - x^\star \rvert \le C \lvert x_t - x^\star \rvert^2$.

*Proof.* Let $e_t = x_t - x^\star$. Expand $f'$ around $x_t$ evaluated at $x^\star$ using Taylor with Lagrange remainder (Proof 3.1 applied to $f'$, $k = 1$):

$$
0 = f'(x^\star) = f'(x_t) - f''(x_t) e_t + \frac{f'''(\xi_t)}{2} e_t^2
$$

for some $\xi_t$ between $x^\star$ and $x_t$. Solve for $f'(x_t)$ and substitute into the Newton update:

$$
e_{t+1} = e_t - \frac{f'(x_t)}{f''(x_t)} = e_t - \frac{f''(x_t) e_t - \frac{1}{2} f'''(\xi_t) e_t^2}{f''(x_t)} = \frac{f'''(\xi_t)}{2 f''(x_t)}\, e_t^2.
$$

By continuity, near $x^\star$ we have $\lvert f''(x_t) \rvert \ge \frac{1}{2}\lvert f''(x^\star) \rvert$ and $\lvert f'''(\xi_t) \rvert \le M$, so $\lvert e_{t+1} \rvert \le C \lvert e_t \rvert^2$ with $C = M/\lvert f''(x^\star) \rvert$. Convergence is quadratic once $\lvert e_0 \rvert \lt 1/C$. $\blacksquare$

*ML footnote.* Pure Newton is rarely used at deep-learning scale (Hessians are huge, and negative curvature breaks the model — Topic 04), but the *local model* viewpoint survives everywhere: Gauss–Newton, K-FAC, L-BFGS, and trust-region SGD variants are all cheapened quadratic models.

## 4. Computational & Algorithmic Insights

### 4.1 Reading error orders off a log-log plot

If the model error behaves as $E(h) = C h^p$, then $\log E = \log C + p \log h$: on log-log axes the error curve is a straight line of slope $p$. Numerically verifying $p = 2$ for the linear model and $p = 3$ for the quadratic model (as done in [`../taylor_approximation.ipynb`](../taylor_approximation.ipynb)) is the standard sanity check that an implementation of derivatives is correct — a wrong gradient shows up instantly as a slope of 1.

The same diagnostic identifies where the asymptotic regime ends: for large $h$ the curve bends away from the straight line (higher-order terms matter), and for tiny $h$ round-off noise floors the plot.

### 4.2 Choosing between the linear and quadratic model in practice

- **Gradient step as model minimization.** Minimizing the descent-lemma upper bound $f(x) + \nabla f(x)^\top h + \frac{L}{2}\lVert h \rVert_2^2$ over $h$ gives $h = -\frac{1}{L}\nabla f(x)$: the classical step size $\eta = 1/L$ is not a tuning trick, it is the argmin of a certified model.
- **Newton step as model minimization.** Minimizing $f(x) + \nabla f(x)^\top h + \frac{1}{2}h^\top H h$ (with $H = \nabla^2 f(x) \succ 0$) gives $H h = -\nabla f(x)$. Solving this linear system with conjugate gradients using only Hessian-vector products (Topic 01, Problem L3.4) is Hessian-free optimization (Martens, 2010).
- **Damping interpolates.** Levenberg–Marquardt replaces $H$ by $H + \mu I$: as $\mu \to \infty$ the step turns into a small gradient step, as $\mu \to 0$ it becomes pure Newton. Trust-region radius updates play the same role.

### 4.3 Taylor models inside modern ML systems

- **Learning-rate schedules**: warmup keeps early steps inside the trust radius of the linear model while curvature statistics are still noisy.
- **Gradient clipping**: a hard cap on $\lVert h \rVert$ is a crude but effective remainder-control device when $L$ is effectively unbounded (exploding gradients in RNNs).
- **Second-order pruning (Optimal Brain Damage/Surgeon)**: the effect of zeroing weight $w_i$ is estimated by the quadratic model, $\Delta L \approx \frac{1}{2} H_{ii} w_i^2$.
- **Influence functions & sharpness measures**: both are Taylor expansions of the loss in parameter or example weight space; "flat minima" arguments compare the $h^\top H h$ term across solutions.

## 5. Real-World Physics & AI/ML Applications

### 5.1 Physics: small oscillations and effective theories

Near a stable equilibrium $x^\star$ of a potential $U$, the force is $-U'(x) \approx -U''(x^\star)(x - x^\star)$: every smooth potential looks like a spring (harmonic oscillator) to second order, with frequency $\omega = \sqrt{U''(x^\star)/m}$. This is the physicist's version of "every nondegenerate minimum is locally quadratic". The pendulum's $\sin\theta \approx \theta - \theta^3/6$ small-angle expansion, relativistic kinetic energy $\sqrt{1+p^2} \approx 1 + \frac{p^2}{2}$, and effective field theories are all Taylor truncations with remainder control.

### 5.2 ML: the second-order picture of a trained model

At a trained minimum $w^\star$ ($\nabla L \approx 0$), the loss is locally

$$
L(w^\star + h) \approx L(w^\star) + \frac{1}{2} h^\top \nabla^2 L(w^\star)\, h.
$$

This single expansion underlies a remarkable range of ML practice: Hessian eigen-spectra measure sharpness and predict generalization gaps; Laplace approximation turns the quadratic model into a Gaussian posterior for Bayesian deep learning; catastrophic-forgetting penalties (EWC) charge new tasks for moving along high-curvature directions; and linear-mode-connectivity studies test where the quadratic picture breaks. When reading any such paper, the first question is always: *which Taylor expansion, at which point, with what remainder assumption?*

## 6. Canonical Literature Mapping & References

| Concept in this notebook | Canonical source | Where |
|---|---|---|
| Second-order approximation of the loss, conditioning | Goodfellow, Bengio & Courville, *Deep Learning* | Ch. 4.3 |
| Taylor's theorem, Lagrange/Cauchy remainders | Spivak, *Calculus*; Apostol, *Calculus Vol. 1* | Ch. 20; Ch. 7 |
| Local models, Newton and trust-region methods | Nocedal & Wright, *Numerical Optimization* | Ch. 2, 4 |
| $L$-smoothness and the descent lemma | Nesterov, *Lectures on Convex Optimization* | Lemma 1.2.3 |
| Newton's method, damping, self-concordance | Boyd & Vandenberghe, *Convex Optimization* | Ch. 9.5 |
| Hessian-free deep learning via quadratic models | Martens, *Deep Learning via Hessian-Free Optimization* (ICML 2010) | §2–4 |
| Smoothness-based analysis of SGD | Bottou, Curtis & Nocedal, *Optimization Methods for Large-Scale ML* (SIAM Review 2018) | §4 |

**Forward pointers**: Topic 03 converts the descent lemma into convergence-rate theorems for gradient descent; Topic 04 studies what the Hessian in the quadratic model says about the global landscape. Runnable error-rate experiments live in the legacy notebook [`../taylor_approximation.ipynb`](../taylor_approximation.ipynb); infinite series and convergence radii are treated in [`../../calculus/09_taylor_and_power_series/`](../../calculus/09_taylor_and_power_series/).